# Data exploration

Quick look at the images before training: class balance, sizes, brightness, preprocessing and augmentation.
Run `python download_data.py` first if `data/raw` is empty.

In [ ]:
import sys, random
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "raw"
SEED = 42
random.seed(SEED); np.random.seed(SEED)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}

def list_split(split, label):
    return sorted(p for p in (DATA / split / label).rglob("*") if p.suffix.lower() in IMG_EXT)

if not any(DATA.rglob("*.jpg")) and not any(DATA.rglob("*.png")):
    raise SystemExit("No data found. Run:  python download_data.py")
source = (DATA / "SOURCE.txt").read_text().strip() if (DATA / "SOURCE.txt").exists() else "unknown"
print("Data source:", source)

## Class balance

In [ ]:
counts = {(s, l): len(list_split(s, l)) for s in ("train", "test") for l in ("perfect", "defective")}
for k, v in counts.items():
    print(f"{k[0]:5s} {k[1]:10s} {v}")

fig, ax = plt.subplots(figsize=(5, 3))
labels = [f"{s}/{l}" for s, l in counts]
ax.bar(labels, counts.values(), color=["tab:green", "tab:red"] * 2)
ax.set_ylabel("images"); ax.set_title("Images per split and class")
plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## Sample images

In [ ]:
def show_samples(n=5):
    rng = random.Random(SEED)
    fig, axes = plt.subplots(2, n, figsize=(2.6 * n, 5.4))
    for row, label in enumerate(("perfect", "defective")):
        files = list_split("train", label)
        for col, path in enumerate(rng.sample(files, min(n, len(files)))):
            img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
            axes[row, col].imshow(img); axes[row, col].axis("off")
            axes[row, col].set_title(label, fontsize=9)
    plt.tight_layout(); plt.show()

show_samples()

## Image sizes and brightness

In [ ]:
shapes, means = {}, {"perfect": [], "defective": []}
for label in means:
    for p in list_split("train", label)[:200]:
        img = cv2.imread(str(p), cv2.IMREAD_COLOR)
        shapes[img.shape] = shapes.get(img.shape, 0) + 1
        means[label].append(img.mean())
print("Original image shapes:", shapes)
for label, vals in means.items():
    print(f"{label:10s} mean brightness {np.mean(vals):6.1f}  (std {np.std(vals):.1f})")

plt.figure(figsize=(5, 3))
for label, color in (("perfect", "tab:green"), ("defective", "tab:red")):
    plt.hist(means[label], bins=20, alpha=0.6, label=label, color=color)
plt.xlabel("mean pixel value"); plt.legend(); plt.title("Brightness per class"); plt.tight_layout(); plt.show()

## Preprocessing

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

path = list_split("train", "defective")[0]
rgb = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
resized = cv2.resize(rgb, (224, 224), interpolation=cv2.INTER_AREA)
normalized = (resized.astype(np.float32) / 255.0 - MEAN) / STD

print("original:", rgb.shape, "| resized:", resized.shape)
print("after normalize: min %.2f max %.2f mean %.2f (roughly centered on 0)" % (normalized.min(), normalized.max(), normalized.mean()))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(rgb); axes[0].set_title(f"original {rgb.shape[1]}x{rgb.shape[0]}")
axes[1].imshow(resized); axes[1].set_title("resized 224x224")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

## Augmentation preview

In [ ]:
def augment(img, rng):
    out = img.copy()
    if rng.random() < 0.5: out = cv2.flip(out, 1)
    if rng.random() < 0.5: out = cv2.flip(out, 0)
    angle = rng.uniform(-15, 15)
    M = cv2.getRotationMatrix2D((112, 112), angle, 1.0)
    out = cv2.warpAffine(out, M, (224, 224), borderMode=cv2.BORDER_REFLECT)
    contrast, brightness = rng.uniform(0.8, 1.2), rng.uniform(-20, 20)
    return np.clip(out.astype(np.float32) * contrast + brightness, 0, 255).astype(np.uint8)

rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
axes[0].imshow(resized); axes[0].set_title("original")
for i in range(1, 6):
    axes[i].imshow(augment(resized, rng)); axes[i].set_title(f"augmented {i}")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

## Things to check
- class balance
- differences between classes other than the defect (brightness, size, background)
- defects still visible at 224x224
- Grad-CAM output in `assets/` after training